# 04 · Gating entropy and collapse diagnostics

Inspect entropy range/distribution and per-seed entropy–loss correlations. Entropy remains positive in these logs; that is evidence against a zero-entropy/binary-collapse signature in recorded telemetry, not proof of general collapse prevention.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
print("Entropy summary:\n",df.entropy.describe())
print("Observed range:",df.entropy.min(),df.entropy.max(),"; binary entropy ceiling ln(2)=",np.log(2))
per=df.groupby("seed").apply(lambda g:pd.Series({"entropy_sd":g.entropy.std(),"min_entropy":g.entropy.min(),"corr_entropy_loss":g.entropy.corr(g.loss)}),include_groups=False)
display(per)
print("Pooled entropy-loss Pearson r:",df.entropy.corr(df.loss))
assert (df.entropy>0).all() and np.isfinite(df.entropy).all()
print("Check passed: no exact zero-entropy observations.")

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,3.7))
ax[0].hist(df.entropy,bins=35,color="#5b8d67"); ax[0].set(title="Entropy distribution",xlabel="entropy",ylabel="steps")
ax[1].scatter(df.entropy,df.loss,s=8,alpha=.35); ax[1].set(title=f"Entropy vs loss (r={df.entropy.corr(df.loss):.3f})",xlabel="entropy",ylabel="loss")
plt.tight_layout()

The reported correlation is descriptive and pooled telemetry points are repeated measurements. Per-seed correlations and entropy variation are more informative than interpreting the pooled coefficient causally.